# Challenge 04 -- Deploy a Hosted Weather Agent on Foundry (SOLUTION)

This is the **completed solution** for Challenge 04. All blanks are filled in.

## Learning Objectives
- Understand the difference between prompt-based and hosted agents
- Build a hosted agent using the **Agent Framework SDK** with a custom tool
- Write the project files: `main.py`, `agent.yaml`, `Dockerfile`, and `requirements.txt`
- Test the agent locally before deploying
- Deploy to Foundry Agent Service via the Python SDK and invoke the hosted endpoint

---
## Setup: Install Dependencies

In [ ]:
%pip install pyyaml requests python-dotenv "azure-ai-projects>=2.1.0" azure-identity --quiet

In [ ]:
# Verify critical packages imported successfully
try:
    import yaml
    import requests
    from azure.identity import DefaultAzureCredential
    from azure.ai.projects import AIProjectClient
    print("[OK] All required packages imported successfully")
except ImportError as e:
    print(f"[FAIL] Missing package: {e}")
    print("Re-run the pip install cell above and restart the kernel if needed.")

---
## Setup: Configure Your Project

Set your Foundry project endpoint, model deployment name, and Azure Container Registry name below.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
load_dotenv()

# --- Your project settings ---
endpoint = "https://new-agent-test-ai.services.ai.azure.com/api/projects/WTH-Hosted-Agents"
model = "gpt-4-1"
acr_name = "crxutdtaalmabgq"

# Create the project directory
os.makedirs("weather-agent", exist_ok=True)

# Write the .env file for local testing
with open("weather-agent/.env", "w") as f:
    f.write(f"FOUNDRY_PROJECT_ENDPOINT={endpoint}\n")
    f.write(f"AZURE_AI_MODEL_DEPLOYMENT_NAME={model}\n")

print(f"[OK] Project directory created: weather-agent/")
print(f"[OK] .env configured:")
print(f"     Endpoint: {endpoint}")
print(f"     Model:    {model}")
print(f"     ACR:      {acr_name}")

In [ ]:
# Validate configuration and Azure credentials
from azure.identity import DefaultAzureCredential

errors = []

if not endpoint.startswith("https://"):
    errors.append("[FAIL] Endpoint should start with 'https://'")
elif "/api/projects/" not in endpoint:
    errors.append("[WARN] Endpoint may be missing '/api/projects/<name>' suffix")

if "." in acr_name or "/" in acr_name:
    errors.append("[FAIL] ACR name should be just the registry name (e.g. 'myregistry'), not the full URL")

try:
    cred = DefaultAzureCredential()
    token = cred.get_token("https://management.azure.com/.default")
    print("[OK] Azure credentials valid -- DefaultAzureCredential working")
except Exception as e:
    errors.append(f"[FAIL] Azure credentials not found. Run 'az login' first.\n       Error: {e}")

if errors:
    for err in errors:
        print(err)
    print("\nFix the issues above before proceeding.")
else:
    print("[OK] Endpoint format valid")
    print("[OK] ACR name format valid")
    print("[OK] All checks passed -- ready to proceed!")

---
## Part 1: Write the Agent Code (`main.py`) -- SOLUTION

In [ ]:
%%writefile weather-agent/main.py
import os
from typing import Annotated

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from pydantic import Field

load_dotenv()


# --- Define the weather tool ---

@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The city or location to get weather for")]
) -> str:
    '''Get the current weather for a given location.'''
    weather_data = {
        "seattle": "55F, cloudy with light rain",
        "new york": "72F, sunny with a few clouds",
        "london": "60F, overcast with drizzle",
        "tokyo": "80F, warm and humid",
        "paris": "63F, clear skies",
    }
    result = weather_data.get(location.lower(), f"65F, partly cloudy in {location}")
    return f"The weather in {location} is: {result}"


# --- Create the Foundry chat client ---

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=DefaultAzureCredential(),
)


# --- Create the agent ---

agent = Agent(
    client=client,
    instructions=(
        "You are a helpful weather assistant. When a user asks about the weather, "
        "use the get_weather tool to look up the current conditions. Always mention "
        "the location in your response and provide a concise, friendly summary. "
        "If the user asks about a location not in your data, provide the default "
        "conditions and let them know the data is simulated."
    ),
    tools=[get_weather],
    default_options={"store": False},
)


# --- Start the hosted server ---

server = ResponsesHostServer(agent)
server.run()

---
## Part 2: Write the Agent Definition (`agent.yaml`) -- SOLUTION

The `agent.yaml` tells Foundry how to run your container -- what protocol it exposes, what resources it needs, and what environment variables to inject.

In [ ]:
%%writefile weather-agent/agent.yaml
kind: hosted
name: weather-agent
protocols:
  - protocol: responses
    version: 1.0.0
resources:
  cpu: "0.5"
  memory: 1Gi
environment_variables:
  - name: AZURE_AI_MODEL_DEPLOYMENT_NAME
    value: gpt-4-1

---
## Part 3: Write Supporting Files

In [ ]:
%%writefile weather-agent/requirements.txt
agent-framework>=1.2.2
agent-framework-foundry-hosting
azure-identity
python-dotenv
pydantic

In [ ]:
%%writefile weather-agent/Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY . user_agent/
WORKDIR /app/user_agent

RUN if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

EXPOSE 8088

CMD ["python", "main.py"]

In [ ]:
%%writefile weather-agent/.dockerignore
.venv
__pycache__
*.pyc
*.pyo
*.pyd
.Python
.env

---
## Validate Your Code

In [ ]:
import py_compile
import yaml

try:
    py_compile.compile("weather-agent/main.py", doraise=True)
    print("[OK] main.py -- syntax OK")
except py_compile.PyCompileError as e:
    print(f"[FAIL] main.py -- syntax error:\n{e}")

try:
    with open("weather-agent/agent.yaml") as f:
        data = yaml.safe_load(f)
    assert data.get("kind") == "hosted", "kind should be 'hosted'"
    assert data.get("protocols"), "Missing 'protocols' field"
    print(f"[OK] agent.yaml -- valid (kind={data['kind']}, name={data.get('name')})")
except Exception as e:
    print(f"[FAIL] agent.yaml -- error: {e}")

print("\nProject structure:")
for fname in sorted(os.listdir("weather-agent")):
    if not fname.startswith("__"):
        size = os.path.getsize(f"weather-agent/{fname}")
        print(f"   {fname} ({size} bytes)")

---
## Part 4: Test the Agent Locally

> **Prerequisite:** You must be logged in to Azure (`az login`) for `DefaultAzureCredential` to work.

In [ ]:
import subprocess
import time
import requests

process = subprocess.Popen(
    ["python", "main.py"],
    cwd="weather-agent",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("Starting agent server...")
ready = False
for i in range(30):
    try:
        r = requests.get("http://localhost:8088/readiness")
        if r.status_code == 200:
            print(f"[OK] Agent server is ready! (took {i + 1}s)")
            ready = True
            break
    except requests.ConnectionError:
        pass
    time.sleep(1)

if not ready:
    print("[FAIL] Server did not start in 30s. Check for errors:")
    output = process.stdout.read(4096).decode(errors="replace")
    print(output)

### Send a Weather Query

In [ ]:
import json

url = "http://localhost:8088/responses"

headers = {"Content-Type": "application/json"}
payload = {
    "input": "What's the weather like in Seattle?",
    "model": "gpt-4-1",
}

response = requests.post(url, headers=headers, json=payload)
data = response.json()

print("Raw response (first 500 chars):")
print(json.dumps(data, indent=2)[:500])

print("\nAgent says:")
for item in data.get("output", []):
    if item.get("type") == "message":
        for content in item.get("content", []):
            if content.get("type") == "output_text":
                print(content["text"])

### Multi-Turn Conversation

In [ ]:
previous_response_id = data.get("id")

payload2 = {
    "input": "How about Tokyo? Is it warmer there?",
    "model": "gpt-4-1",
    "previous_response_id": previous_response_id,
}

response2 = requests.post(url, headers=headers, json=payload2)
data2 = response2.json()

print("Agent says (multi-turn):")
for item in data2.get("output", []):
    if item.get("type") == "message":
        for content in item.get("content", []):
            if content.get("type") == "output_text":
                print(content["text"])

---
## Stop the Local Server

In [ ]:
if process and process.poll() is None:
    process.terminate()
    process.wait(timeout=5)
    print("[OK] Agent server stopped.")
else:
    print("[INFO] Server was not running.")

---
## Part 5: Build Image and Deploy to Foundry

Deployment has three steps:
1. **Build** the container image on Azure Container Registry (no Docker needed locally)
2. **Deploy** the agent version via the Python SDK
3. **Poll** until the agent status is `active`

### Step 1: Build the container image on ACR

In [ ]:
# Build the Docker image remotely on ACR -- no local Docker needed!
!az acr build --registry {acr_name} --image weather-agent:v1 --platform linux/amd64 weather-agent/

In [ ]:
# Verify the image was built successfully
import subprocess, json

result = subprocess.run(
    ["az", "acr", "repository", "show-tags", "--name", acr_name, "--repository", "weather-agent", "--output", "json"],
    capture_output=True, text=True
)
if result.returncode == 0:
    tags = json.loads(result.stdout)
    if "v1" in tags:
        print(f"[OK] Image weather-agent:v1 exists in ACR '{acr_name}'")
        print(f"     Available tags: {tags}")
    else:
        print(f"[FAIL] Tag 'v1' not found. Available tags: {tags}")
else:
    print(f"[FAIL] Could not query ACR: {result.stderr}")

### Step 2: Deploy the agent via Python SDK

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import HostedAgentDefinition, ProtocolVersionRecord, AgentProtocol
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
project = AIProjectClient(
    endpoint=endpoint,
    credential=credential,
    allow_preview=True,
)

image_url = f"{acr_name}.azurecr.io/weather-agent:v1"

agent_version = project.agents.create_version(
    agent_name="weather-agent",
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="1.0.0")
        ],
        cpu="0.5",
        memory="1Gi",
        image=image_url,
        environment_variables={
            "AZURE_AI_MODEL_DEPLOYMENT_NAME": model,
        },
    ),
)

print(f"[OK] Agent created: {agent_version.name}, version: {agent_version.version}")

### Step 3: Poll until the agent is active

In [ ]:
import time

while True:
    info = project.agents.get_version(
        agent_name="weather-agent",
        agent_version=agent_version.version,
    )
    status = info["status"]
    print(f"Status: {status}")
    if status == "active":
        print("[OK] Agent is deployed and ready!")
        break
    elif status == "failed":
        print(f"[FAIL] Provisioning failed: {info.get('error', 'unknown')}")
        break
    time.sleep(10)

---
## Part 6: Invoke the Deployed Agent

Now send a request to the hosted agent running on Foundry (not localhost).

In [ ]:
openai_client = project.get_openai_client(agent_name="weather-agent")

response = openai_client.responses.create(
    input="What is the weather like in Paris right now?",
)

print("Deployed agent says:")
print(response.output_text)

In [ ]:
response2 = openai_client.responses.create(
    input="How about London?",
)

print("Deployed agent says:")
print(response2.output_text)

In [ ]:
# Validate the responses
for i, resp in enumerate([response, response2], 1):
    if hasattr(resp, 'output_text') and resp.output_text:
        print(f"[OK] Response {i}: got {len(resp.output_text)} chars")
    else:
        print(f"[FAIL] Response {i}: no text received")

---
## Verify in the Foundry Portal

Open the [Microsoft Foundry portal](https://ai.azure.com) and confirm:
1. Navigate to your project > **Agents** section
2. Your Weather Agent appears in the agents list
3. The status shows **Active**

In [ ]:
print("=" * 55)
print("CHALLENGE 04 -- COMPLETION CHECKLIST")
print("=" * 55)
print()
print("[x]  Project structure: main.py, agent.yaml, Dockerfile, requirements.txt")
print("[x]  Weather tool defined with @tool decorator")
print("[x]  Agent uses DefaultAzureCredential (no API keys)")
print("[x]  Agent tested locally on port 8088")
print("[x]  Multi-turn conversation works")
print("[x]  Container image built on ACR")
print("[x]  Agent deployed via Python SDK")
print("[x]  Agent status is Active")
print("[x]  Invoked deployed agent successfully")

---
## Key Takeaways

| Concept | Prompt-Based Agents (Ch 01-03) | Hosted Agents (Ch 04) |
|---|---|---|
| **What you write** | System prompt + tool config | Full runtime code (`main.py`) |
| **Where it runs** | Foundry-managed (serverless) | Your container on Foundry compute |
| **Framework** | Azure AI Agent SDK only | Any framework (Agent Framework, LangGraph, custom) |
| **Protocol** | SDK threads/runs API | HTTP -- Responses or Invocations protocol |
| **Identity** | Shared project identity | Dedicated Entra ID per agent |
| **Deployment** | `AgentsClient.create_agent()` | SDK `create_version()` or `azd deploy` |
| **Best for** | Quick prototyping, simple agents | Production agents, custom logic, multi-framework |

---
## Cleanup

Delete the agent when finished:
```python
project.agents.delete(agent_name="weather-agent")
```